<a href="https://colab.research.google.com/github/d-noe/NLP_DH_F26/blob/main/code/3_bert_training/Hands_on_3_SS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-On 3: Semantic Shifts


<p align="center">
  <img alt="Widnes Examiner (1883)" src="https://panoptes-uploads.zooniverse.org/subject_location/421af65f-ab5d-4977-9186-65d7f5938902.png" width="54%">
&nbsp; &nbsp; &nbsp; &nbsp;
  <img alt="Widnes Examiner (1916)" src="https://panoptes-uploads.zooniverse.org/subject_location/775591a5-87a9-426d-af8c-08f130a397d8.png" width="41%">
</p>

Let's explore diachronic vector representations, using OCRed historical newspaper excerpts from the [Living with Machines program](https://livingwithmachines.ac.uk/).

Learn more on [Semantic Change on Wikipedia](https://en.wikipedia.org/wiki/Semantic_change).


In [ ]:
#@title A glimpse at the data:


from IPython.core.display import display, HTML
import urllib.request

fp = urllib.request.urlopen("https://raw.githubusercontent.com/d-noe/NLP_DH_F26/refs/heads/main/props/lwm_advanced_trolley.html")
mybytes = fp.read()

mystr = mybytes.decode("utf8")
fp.close()

display(HTML(mystr))

## Set-up

Feel free to jump directly to the next section if you run the notebook locally and that you already have a ready-to-use environment.

Install and import necessary Python libraries and modules.

This notebook will mainly rely on [`transformers` Python library](https://huggingface.co/docs/transformers/installation), we will also use [`scikit-learn`](https://scikit-learn.org/stable/install.html) for dimensionality reduction, and [`altair`](https://altair-viz.github.io/) for visualisation.

In [ ]:
! pip install transformers

In [ ]:
# For BERT -> use lighter version DistilBert
from transformers import DistilBertTokenizerFast, DistilBertModel

# For data manipulation and analysis
import pandas as pd
pd.options.display.max_colwidth = 200
import numpy as np
from sklearn.decomposition import PCA

# For interactive data visualization
import altair as alt

In [ ]:
# are 'GPU's available?
# If using Colab, you can change your runtime to access GPUs
# but it is not really needed here, computations are not too long on CPUs
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

## Familiarize with the data


In the context of this experiment, the data (originally from [here](https://livingwithmachines.ac.uk/)) is presented in the `.csv`, which can be conviniently loaded as a `pandas.DataFrame`. This file contains the following columns:
- `text` [string]: Historical Newspaper article excerpt (OCRed)
- `target` [string]: the word of interest `text` (example: *'trolley'*)
- ...

**Propositions**

- Select a word from the following list: `trolley`, `car`, `coach`, `bike`.
- Filter the dataset to contain only rows related to the selected word `target`
- Have a look at the data:
  - What are texts like?
  - What is the time distribution of the included articles?
  - Where are these articles from?
  - What are the annotation labels?
- Find a way to filter down the data
  - If dealing with technical/computational limitations: reduce the number of samples to limit the computation costs
  - Reflect on the impact that it could have during downstream analysis and keep this in mind
  - Plus, here we have the chance to have annotated data (see `annot` column), why not use the annotations?


In [ ]:
!wget https://raw.githubusercontent.com/d-noe/NLP_DH_F26/refs/heads/main/code/scripts/helpers.py
from helpers import load_csv_from_github

# Load the data
df = load_csv_from_github("data/word_sense/lwm_annot.csv")
print(f"The DataFrame contains {len(df)} rows.")

# Display 5 random rows
df.sample(5)

In [ ]:
# TODO: run you code here, feel free to add cells

In [ ]:
# By now, you should have filtered down the `df` to the samples you want to analyze.
# Let's store it in `df_filtered`

# Un-comment and store it here:
# df_filtered =

## Produce Vector Representations

Now, that you are all set with your data, let's focus on the core of this approach: producing embeddings from the data using a pre-trained LM.

First load the model and tokenizer, then produce the vectors and retrieve the vectors associated with the word that you are studying.

Earlier, in the tutorial, we used a quick and dirty heuristic to retrieve the vector representations associated with our focus word. That was not very convincing nor robust... Let's address this shortcoming: define a `get_vector_representations` function that can extract the embeddings for any word!

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained("distilbert-base-uncased").to(DEVICE)

In [ ]:
# Define your function(s) here!

def get_vector_representations(
    model,
    tokenizer,
    texts:list[str],
    focus_word:str,
    **kwargs
):
  """
  Input
    - model:
        pre-trained LM, used to embed texts
    - tokenizer:
        associated tokenizer
    - texts: [list[str]]
        texts to be embedded
    - focus_word: [str]
        word of interest: get the representations at this position
  Output
    - embeddings: [array-like]
        embeddings for the focus_word in each instance of 'texts'
  """


  raise NotImplementedError("You need to implement the 'get_vector_representations' function first!")
  return embeddings


In [ ]:
list_of_texts = df_filtered["text"].to_list()

In [ ]:
embeddings = get_vector_representations(
    model=model,
    tokenizer=tokenizer,
    texts=list_of_texts,
    focus_word=focus_word,
)

### Visualise your results

Apply dimensionality reduction on the embeddings, and visualize their distribution in a 2D space.

In [ ]:
# perform dimensionality reduction on `embeddings`
# + store in a dataframe to be able to visualize directly below

# 1. Compute 'reduced_vectors'




In [ ]:
# 2. Store in a DataFrame

# df_plot = df_filtered.copy()
# df_plot["dim_1"] = reduced_vectors[:,0]
# df_plot["dim_2"] = reduced_vectors[:,1]

In [ ]:
chart = alt.Chart(
    df_plot,
    title=f"Word PCA Distribution: {focus_word}"
).mark_circle(size=200).encode(
    alt.X('dim_1',
        scale=alt.Scale(zero=False)
    ),
    y="dim_2",
    color= "label_sense",
    tooltip=['target', 'label_sense', 'text'],
    ).interactive().properties(
    width=500,
    height=500
)
chart

### Analyse

Ideas:

- Find if their is really a *drift* within the representation space.
- Recall last week's semantic shift analysis with `Word2Vec`, how is that approach different? How do the results compare?



## Want more?

**Challenges:**
1. Can you find an embedding dimension that encodes time across concepts?
2. Explore different models (other than `distilbert`), how does it impact the results?
3. We used the "last hidden state" vector at the position of the `focus_word`, is this the best possible choice? Try to extract different vector representations (e.g.; from other layer(s), using mean-pooling, or concatenation) and study the influence on the visualisation / analysis.